# mini_ir_engine build an inverted index

## goal
build a simple inverted index from a small document collection and understand how documents,terms,and posting lists are represented in code.

## pipeline
documents -> tokenization -> inverted index


**Status:** Implemented educational prototype. This notebook demonstrates indexing, Boolean retrieval, TF-IDF ranking, and postings compression on seven illustrative documents.

In [22]:
documents = {
    1: "information retrieval uses inverted indexes",
    2: "index compression reduces storage",
    3: "retrieval systems use postings lists",
    4: "inverted indexes support fast retrieval",
    5: "compression can reduce postings size",
    6: "retrieval retrieval systems improve search",
    7: "compression compression compression reduces index storage"
}

documents

{1: 'information retrieval uses inverted indexes',
 2: 'index compression reduces storage',
 3: 'retrieval systems use postings lists',
 4: 'inverted indexes support fast retrieval',
 5: 'compression can reduce postings size',
 6: 'retrieval retrieval systems improve search',
 7: 'compression compression compression reduces index storage'}

In [23]:
text=documents[1]
tokens=text.split()
tokens

['information', 'retrieval', 'uses', 'inverted', 'indexes']

In [24]:
import re
def tokenize(text):
    text=text.lower()
    tokens=re.findall(r"[a-z]+",text)
    return tokens




In [25]:
tokenized_documents={}
for doc_id,text in documents.items():
    tokenized_documents[doc_id]=tokenize(text)

tokenized_documents



{1: ['information', 'retrieval', 'uses', 'inverted', 'indexes'],
 2: ['index', 'compression', 'reduces', 'storage'],
 3: ['retrieval', 'systems', 'use', 'postings', 'lists'],
 4: ['inverted', 'indexes', 'support', 'fast', 'retrieval'],
 5: ['compression', 'can', 'reduce', 'postings', 'size'],
 6: ['retrieval', 'retrieval', 'systems', 'improve', 'search'],
 7: ['compression',
  'compression',
  'compression',
  'reduces',
  'index',
  'storage']}

In [26]:
inverted_index={}
for doc_id,tokens in tokenized_documents.items():
    print(doc_id,tokens)

1 ['information', 'retrieval', 'uses', 'inverted', 'indexes']
2 ['index', 'compression', 'reduces', 'storage']
3 ['retrieval', 'systems', 'use', 'postings', 'lists']
4 ['inverted', 'indexes', 'support', 'fast', 'retrieval']
5 ['compression', 'can', 'reduce', 'postings', 'size']
6 ['retrieval', 'retrieval', 'systems', 'improve', 'search']
7 ['compression', 'compression', 'compression', 'reduces', 'index', 'storage']


In [27]:
for doc_id,tokens in tokenized_documents.items():
    for term in tokens:
        print(term,doc_id)

information 1
retrieval 1
uses 1
inverted 1
indexes 1
index 2
compression 2
reduces 2
storage 2
retrieval 3
systems 3
use 3
postings 3
lists 3
inverted 4
indexes 4
support 4
fast 4
retrieval 4
compression 5
can 5
reduce 5
postings 5
size 5
retrieval 6
retrieval 6
systems 6
improve 6
search 6
compression 7
compression 7
compression 7
reduces 7
index 7
storage 7


In [28]:
inverted_index={}
for doc_id,tokens in tokenized_documents.items():
    for term in tokens:
        if term not in inverted_index:
            inverted_index[term]=[doc_id]
        else:
            inverted_index[term].append(doc_id)

inverted_index


{'information': [1],
 'retrieval': [1, 3, 4, 6, 6],
 'uses': [1],
 'inverted': [1, 4],
 'indexes': [1, 4],
 'index': [2, 7],
 'compression': [2, 5, 7, 7, 7],
 'reduces': [2, 7],
 'storage': [2, 7],
 'systems': [3, 6],
 'use': [3],
 'postings': [3, 5],
 'lists': [3],
 'support': [4],
 'fast': [4],
 'can': [5],
 'reduce': [5],
 'size': [5],
 'improve': [6],
 'search': [6]}

In [29]:
inverted_index={}
for doc_id,tokens in tokenized_documents.items():
    for term in set(tokens):
        if term not in inverted_index:
            inverted_index[term]=[doc_id]
        else:
            inverted_index[term].append(doc_id)

inverted_index

{'information': [1],
 'uses': [1],
 'retrieval': [1, 3, 4, 6],
 'inverted': [1, 4],
 'indexes': [1, 4],
 'storage': [2, 7],
 'reduces': [2, 7],
 'compression': [2, 5, 7],
 'index': [2, 7],
 'postings': [3, 5],
 'use': [3],
 'lists': [3],
 'systems': [3, 6],
 'support': [4],
 'fast': [4],
 'can': [5],
 'reduce': [5],
 'size': [5],
 'improve': [6],
 'search': [6]}

In [30]:
print(inverted_index.get("retrieval",[]))
print(inverted_index.get("apple",[]))

[1, 3, 4, 6]
[]


In [31]:
def search_single_term(query):
    tokens=tokenize(query)
    
    if len(tokens)==0:
        return[]
    term=tokens[0]
    return inverted_index.get(term,[])
    

In [32]:
print(search_single_term("retrieval"))
print(search_single_term("Retrieval!"))
print(search_single_term("apple"))

[1, 3, 4, 6]
[1, 3, 4, 6]
[]


In [33]:
def intersect(postings1,postings2):
    result=[]
    i=0 
    j=0
    while i<len(postings1) and j<len(postings2):
        if postings1[i]==postings2[j]:
            result.append(postings1[i])
            i+=1
            j+=1
        elif postings1[i]<postings2[j]:
            i+=1
        else:
            j+=1
    return result

In [34]:
intersect([1,4],[1,3,4])

[1, 4]

In [35]:
def search_and(query):
    terms=tokenize(query)
    if len(terms)==0:
        return []
    result=inverted_index.get(terms[0],[])
    for term in terms[1:]:
        postings=inverted_index.get(term,[])
        result=intersect(result,postings)
    return result

In [36]:
print(search_and("inverted retrieval"))
print(search_and("Retrieval postings"))
print(search_and("retrieval apple"))

[1, 4]
[3]
[]


In [37]:
print(inverted_index["retrieval"])
print(inverted_index["compression"])

[1, 3, 4, 6]
[2, 5, 7]


In [42]:
term_frequencies={}
for doc_id,tokens in tokenized_documents.items():
    term_frequencies[doc_id]={}
    for term in tokens:
        if term not in term_frequencies[doc_id]:
            term_frequencies[doc_id][term]=1
        else:
            term_frequencies[doc_id][term]+=1

In [43]:
print(term_frequencies[6])
print(term_frequencies[7])

{'retrieval': 2, 'systems': 1, 'improve': 1, 'search': 1}
{'compression': 3, 'reduces': 1, 'index': 1, 'storage': 1}


In [44]:
df_compression=len(inverted_index["compression"])
df_retrieval=len(inverted_index["retrieval"])
print(df_compression)
print(df_retrieval)

3
4


In [45]:
import math

In [46]:
N=len(documents)
idf_compression=math.log10(N/df_compression)
idf_retrieval=math.log10(N/df_retrieval)
print(idf_compression)
print(idf_retrieval)

0.36797678529459443
0.24303804868629444


In [47]:
idf={}
for term,postings in inverted_index.items():
    df=len(postings)
    idf[term]=math.log10(N/df)

idf

{'information': 0.8450980400142568,
 'uses': 0.8450980400142568,
 'retrieval': 0.24303804868629444,
 'inverted': 0.5440680443502757,
 'indexes': 0.5440680443502757,
 'storage': 0.5440680443502757,
 'reduces': 0.5440680443502757,
 'compression': 0.36797678529459443,
 'index': 0.5440680443502757,
 'postings': 0.5440680443502757,
 'use': 0.8450980400142568,
 'lists': 0.8450980400142568,
 'systems': 0.5440680443502757,
 'support': 0.8450980400142568,
 'fast': 0.8450980400142568,
 'can': 0.8450980400142568,
 'reduce': 0.8450980400142568,
 'size': 0.8450980400142568,
 'improve': 0.8450980400142568,
 'search': 0.8450980400142568}

In [51]:
tfidf_weights={}
for doc_id,term_counts in term_frequencies.items():
    tfidf_weights[doc_id]={}
    for term,tf in term_counts.items():
        tfidf_weights[doc_id][term]=tf*idf[term]

tfidf_weights

{1: {'information': 0.8450980400142568,
  'retrieval': 0.24303804868629444,
  'uses': 0.8450980400142568,
  'inverted': 0.5440680443502757,
  'indexes': 0.5440680443502757},
 2: {'index': 0.5440680443502757,
  'compression': 0.36797678529459443,
  'reduces': 0.5440680443502757,
  'storage': 0.5440680443502757},
 3: {'retrieval': 0.24303804868629444,
  'systems': 0.5440680443502757,
  'use': 0.8450980400142568,
  'postings': 0.5440680443502757,
  'lists': 0.8450980400142568},
 4: {'inverted': 0.5440680443502757,
  'indexes': 0.5440680443502757,
  'support': 0.8450980400142568,
  'fast': 0.8450980400142568,
  'retrieval': 0.24303804868629444},
 5: {'compression': 0.36797678529459443,
  'can': 0.8450980400142568,
  'reduce': 0.8450980400142568,
  'postings': 0.5440680443502757,
  'size': 0.8450980400142568},
 6: {'retrieval': 0.4860760973725889,
  'systems': 0.5440680443502757,
  'improve': 0.8450980400142568,
  'search': 0.8450980400142568},
 7: {'compression': 1.1039303558837834,
  'red

In [54]:
document_lengths={}
for doc_id,weights in tfidf_weights.items():
    sum_of_squares=0;
    for weight in weights.values():
        sum_of_squares+=weight**2
    document_lengths[doc_id]=math.sqrt(sum_of_squares)
document_lengths

{1: 1.4420363939052945,
 2: 1.0116506438317254,
 3: 1.4420363939052945,
 4: 1.4420363939052945,
 5: 1.6043656201460728,
 6: 1.4002361957155576,
 7: 1.4514449150040443}

In [55]:
normalized_tfidf = {}
for doc_id, weights in tfidf_weights.items():
    normalized_tfidf[doc_id] = {}
    for term, weight in weights.items():
        normalized_tfidf[doc_id][term] = weight / document_lengths[doc_id]
normalized_tfidf

{1: {'information': 0.5860448762500223,
  'retrieval': 0.16853808247384353,
  'uses': 0.5860448762500223,
  'inverted': 0.37729147936193297,
  'indexes': 0.37729147936193297},
 2: {'index': 0.537802301286109,
  'compression': 0.3637389918528065,
  'reduces': 0.537802301286109,
  'storage': 0.537802301286109},
 3: {'retrieval': 0.16853808247384353,
  'systems': 0.37729147936193297,
  'use': 0.5860448762500223,
  'postings': 0.37729147936193297,
  'lists': 0.5860448762500223},
 4: {'inverted': 0.37729147936193297,
  'indexes': 0.37729147936193297,
  'support': 0.5860448762500223,
  'fast': 0.5860448762500223,
  'retrieval': 0.16853808247384353},
 5: {'compression': 0.22935968003421267,
  'can': 0.526749033638176,
  'reduce': 0.526749033638176,
  'postings': 0.3391172420540524,
  'size': 0.526749033638176},
 6: {'retrieval': 0.34713864622260476,
  'systems': 0.3885544781766212,
  'improve': 0.6035396332419399,
  'search': 0.6035396332419399},
 7: {'compression': 0.7605733737960751,
  'red

In [56]:
query="compression index"
query_terms=tokenize(query)
query_terms

['compression', 'index']

In [57]:
query_tf={}
for term in query_terms:
    if term not in query_tf:
        query_tf[term]=1
    else:
        query_tf[term]+=1
query_tf

{'compression': 1, 'index': 1}

In [58]:
query_tfidf = {}
for term, tf in query_tf.items():
    if term in idf:
        query_tfidf[term] = tf * idf[term]
query_tfidf

{'compression': 0.36797678529459443, 'index': 0.5440680443502757}

In [59]:
query_length = math.sqrt(sum(weight ** 2 for weight in query_tfidf.values()))
normalized_query = {}
for term, weight in query_tfidf.items():
    normalized_query[term] = weight / query_length
normalized_query

{'compression': 0.5602370393246845, 'index': 0.8283323365465772}

In [60]:
scores = {}
for doc_id, doc_vector in normalized_tfidf.items():
    score = 0
    for term, query_weight in normalized_query.items():
        doc_weight = doc_vector.get(term, 0)
        score += query_weight * doc_weight
    scores[doc_id] = score
scores

{1: 0.0,
 2: 0.6492590927070108,
 3: 0.0,
 4: 0.0,
 5: 0.12849578808282425,
 6: 0.0,
 7: 0.7365982804900231}

In [61]:
ranked_results = sorted(
    scores.items(),
    key=lambda item: item[1],
    reverse=True
)
ranked_results

[(7, 0.7365982804900231),
 (2, 0.6492590927070108),
 (5, 0.12849578808282425),
 (1, 0.0),
 (3, 0.0),
 (4, 0.0),
 (6, 0.0)]

In [62]:
for rank,(doc_id,score) in enumerate(ranked_results,start=1):
    print(rank,doc_id,round(score,4),documents[doc_id])

1 7 0.7366 compression compression compression reduces index storage
2 2 0.6493 index compression reduces storage
3 5 0.1285 compression can reduce postings size
4 1 0.0 information retrieval uses inverted indexes
5 3 0.0 retrieval systems use postings lists
6 4 0.0 inverted indexes support fast retrieval
7 6 0.0 retrieval retrieval systems improve search


In [63]:
nonzero_results = []
for doc_id, score in ranked_results:
    if score > 0:
        nonzero_results.append((doc_id, score))
nonzero_results

[(7, 0.7365982804900231), (2, 0.6492590927070108), (5, 0.12849578808282425)]

In [64]:
k = 2
top_k = nonzero_results[:k]
top_k

[(7, 0.7365982804900231), (2, 0.6492590927070108)]

In [65]:
def d_gap_encode(postings):
    if len(postings) == 0:
        return []
    gaps = [postings[0]]
    for i in range(1, len(postings)):
        gap = postings[i] - postings[i - 1]
        gaps.append(gap)
    return gaps

retrieval_postings = inverted_index["retrieval"]
d_gap_encode(retrieval_postings)

[1, 2, 1, 2]

In [66]:
def d_gap_decode(gaps):
    if len(gaps) == 0:
        return []
    postings = [gaps[0]]
    for i in range(1, len(gaps)):
        doc_id = postings[-1] + gaps[i]
        postings.append(doc_id)
    return postings

gaps = d_gap_encode(retrieval_postings)
decoded = d_gap_decode(gaps)
print(gaps)
print(decoded)

[1, 2, 1, 2]
[1, 3, 4, 6]


In [67]:
def vb_encode_number(n):
    bytes_list = []
    while True:
        bytes_list.insert(0, n % 128)
        if n < 128:
            break
        n = n // 128
    bytes_list[-1] += 128
    return bytes_list

print(vb_encode_number(1))
print(vb_encode_number(127))
print(vb_encode_number(128))

[129]
[255]
[1, 128]


In [68]:
def vb_encode(numbers):
    encoded = []
    for n in numbers:
        encoded.extend(vb_encode_number(n))
    return encoded

gaps = d_gap_encode(retrieval_postings)
encoded = vb_encode(gaps)
print(gaps)
print(encoded)

[1, 2, 1, 2]
[129, 130, 129, 130]


In [69]:
def vb_decode(encoded):
    numbers = []
    n = 0
    for byte in encoded:
        if byte < 128:
            n = 128 * n + byte
        else:
            n = 128 * n + (byte - 128)
            numbers.append(n)
            n = 0
    return numbers

decoded_gaps = vb_decode(encoded)
print(encoded)
print(decoded_gaps)

[129, 130, 129, 130]
[1, 2, 1, 2]


In [70]:
decoded_postings = d_gap_decode(decoded_gaps)
print(decoded_postings)
print(decoded_postings == retrieval_postings)

[1, 3, 4, 6]
True


In [71]:
compressed_postings = {}
for term, postings in inverted_index.items():
    gaps = d_gap_encode(postings)
    encoded = vb_encode(gaps)
    compressed_postings[term] = encoded

compressed_postings

{'information': [129],
 'uses': [129],
 'retrieval': [129, 130, 129, 130],
 'inverted': [129, 131],
 'indexes': [129, 131],
 'storage': [130, 133],
 'reduces': [130, 133],
 'compression': [130, 131, 130],
 'index': [130, 133],
 'postings': [131, 130],
 'use': [131],
 'lists': [131],
 'systems': [131, 131],
 'support': [132],
 'fast': [132],
 'can': [133],
 'reduce': [133],
 'size': [133],
 'improve': [134],
 'search': [134]}

In [72]:
print(inverted_index["retrieval"])
print(compressed_postings["retrieval"])

[1, 3, 4, 6]
[129, 130, 129, 130]


In [75]:
raw_bytes=0
for postings in inverted_index.values():
    raw_bytes+=len(postings)*4
raw_bytes

128

In [76]:
compressed_bytes=0
for encoded in compressed_postings.values():
    compressed_bytes+=len(encoded)
compressed_bytes

32

In [77]:
compression_ratio=compressed_bytes/raw_bytes
print(raw_bytes)
print(compressed_bytes)

128
32


In [78]:
print(compression_ratio)

0.25


In [79]:
all_correct = True
for term, original_postings in inverted_index.items():
    encoded = compressed_postings[term]
    decoded_gaps = vb_decode(encoded)
    decoded_postings = d_gap_decode(decoded_gaps)
    if decoded_postings != original_postings:
        all_correct = False
        print("Error:", term)
all_correct

True